# DR Preprocessing — APTOS 2019 Dataset

Builds a clean, reproducible train/val split from the **raw** APTOS 2019 competition data.

**Important:** the competition's own `test.csv` / `test_images/` have no labels (it's the private
leaderboard set) -- they can't be used for validation. All train/val data here comes from
`train.csv` + `train_images/` only, split once with a fixed seed. This is also what fixes the
leakage risk in the old notebook, where the "test set" was topped up with images pulled straight
out of `train.csv` with no guarantee they didn't also end up in the fine-tuning train set.

**Pipeline:**
1. Download the raw competition data from Kaggle
2. Crop + resize + sharpen every image (straight to 528×528, matching EfficientNetB6's input size)
3. One stratified 85/15 train/val split of `train.csv`, fixed random seed
4. Offline data augmentation on the **train split only**, balancing every minority class up to
   the majority class's count
5. Copy the final `train/` and `val/` folders to Google Drive

> Note: this preprocessing function is duplicated from `01_preprocessing_2015.ipynb` for now since
> each notebook needs to stand alone in Colab. When this moves to GitHub, pull both copies into a
> single `src/preprocessing.py` module that both notebooks import.


## 0. Download the dataset from Kaggle

This is a **competition** dataset -- before the download below will work, you must open
https://www.kaggle.com/competitions/aptos2019-blindness-detection/rules in a browser and accept
the competition rules with the same Kaggle account as your API token, otherwise `kaggle
competitions download` returns a 403.

In [4]:
import os
import json
from google.colab import userdata

username = userdata.get("KAGGLE_USERNAME")
key = userdata.get("KAGGLE_KEY")

os.makedirs('/root/.kaggle', exist_ok=True)

# Legacy format
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": username, "key": key}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# New-style single-token format
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(key)
os.chmod('/root/.kaggle/access_token', 0o600)

!pip install -q --upgrade kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 3.9 MB/s eta 0:00:00


In [5]:
!kaggle competitions download -c aptos2019-blindness-detection -p /content/data_2019
!unzip -q -o /content/data_2019/aptos2019-blindness-detection.zip -d /content/data_2019


100% 9.51G/9.51G [01:22<00:00, 123MB/s] 



In [6]:
# Sanity check before hardcoding paths below.
for item in sorted(os.listdir('/content/data_2019')):
    print(item)


aptos2019-blindness-detection.zip
sample_submission.csv
test.csv
test_images
train.csv
train_images


## 1. Imports and preprocessing functions

In [7]:
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import albumentations as A
import shutil

IMG_SIZE = 528  # matches EfficientNetB6 input size used in the training notebook -- no double resizing

APTOS_ROOT = "/content/data_2019"
TRAIN_CSV = os.path.join(APTOS_ROOT, "train.csv")
TRAIN_IMAGES_DIR = os.path.join(APTOS_ROOT, "train_images")


In [8]:
def crop_image_from_gray(img, tol=7):
    """Crop the near-black border around the fundus photo."""
    if img.ndim == 2:
        mask = img > tol
        return img[np.ix_(mask.any(1), mask.any(0))]
    elif img.ndim == 3:
        gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        mask = gray_img > tol
        if mask.any():
            img1 = img[:, :, 0][np.ix_(mask.any(1), mask.any(0))]
            img2 = img[:, :, 1][np.ix_(mask.any(1), mask.any(0))]
            img3 = img[:, :, 2][np.ix_(mask.any(1), mask.any(0))]
            img = np.stack([img1, img2, img3], axis=-1)
        return img
    return img


def preprocess_image(image_path, desired_size=IMG_SIZE):
    """Crop black border, resize, and apply the Ben Graham-style unsharp mask."""
    im = cv2.imread(image_path)
    if im is None:
        return None
    im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
    im = crop_image_from_gray(im)
    if im is None or im.shape[0] == 0 or im.shape[1] == 0:
        return None
    im = cv2.resize(im, (desired_size, desired_size))
    res = cv2.addWeighted(im, 4.5, cv2.GaussianBlur(im, (0, 0), 10), -4, 100)
    return res


## 2. Build the file list from train.csv and split train/val

One stratified 85/15 split, done once, from `train.csv` only -- `test.csv` is never touched since it has no labels. This guarantees no image can appear in both splits.

In [9]:
df = pd.read_csv(TRAIN_CSV)
df["path"] = df["id_code"].apply(lambda x: os.path.join(TRAIN_IMAGES_DIR, f"{x}.png"))

print(f"Total labeled images: {len(df)}")
print(df["diagnosis"].value_counts().sort_index())

train_df, val_df = train_test_split(
    df, test_size=0.15, stratify=df["diagnosis"], random_state=42
)

print(f"\nTrain: {len(train_df)} | Val: {len(val_df)}")


Total labeled images: 3662
diagnosis
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64

Train: 3112 | Val: 550


## 3. Preprocess and save both splits

In [10]:
TRAIN_DIR = "/content/preprocessed_2019/train"
VAL_DIR = "/content/preprocessed_2019/val"


def process_and_save(df_split, out_dir):
    for cls in range(5):
        os.makedirs(os.path.join(out_dir, str(cls)), exist_ok=True)

    for _, row in tqdm(df_split.iterrows(), total=len(df_split)):
        img = preprocess_image(row["path"])
        if img is None:
            continue
        save_path = os.path.join(out_dir, str(row["diagnosis"]), f"{row['id_code']}.png")
        cv2.imwrite(save_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))


process_and_save(train_df, TRAIN_DIR)
process_and_save(val_df, VAL_DIR)


100%|██████████| 550/550 [01:35<00:00,  5.76it/s]


In [11]:
def count_classes(directory):
    counts = {}
    for cls in sorted(os.listdir(directory), key=int):
        class_dir = os.path.join(directory, cls)
        counts[int(cls)] = len(os.listdir(class_dir))
    return counts


print("Train class counts (before augmentation):", count_classes(TRAIN_DIR))
print("Val class counts:  ", count_classes(VAL_DIR))


Train class counts (before augmentation): {0: 1534, 1: 314, 2: 849, 3: 164, 4: 251}
Val class counts:   {0: 271, 1: 56, 2: 150, 3: 29, 4: 44}


## 4. Offline augmentation -- balance minority classes (train split only)

Same approach as the 2015 notebook: every class below the majority class's count gets extra augmented copies generated from its own images, until it matches the majority. `val/` is never touched.

In [12]:
balance_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.4),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=20, p=0.7),
    A.HueSaturationValue(p=0.3),
])

VALID_EXTS = ('.jpg', '.jpeg', '.png')


def augment_and_save(class_dir, num_to_add):
    existing_files = [f for f in os.listdir(class_dir) if f.lower().endswith(VALID_EXTS)]
    for i in tqdm(range(num_to_add)):
        src_name = existing_files[i % len(existing_files)]
        img = cv2.imread(os.path.join(class_dir, src_name))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        augmented = balance_aug(image=img_rgb)['image']
        save_name = f"aug_{i}_{src_name}"
        cv2.imwrite(os.path.join(class_dir, save_name), cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR))


/usr/local/lib/python3.13/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [13]:
train_counts = count_classes(TRAIN_DIR)
majority_count = max(train_counts.values())

for cls, count in train_counts.items():
    deficit = majority_count - count
    if deficit > 0:
        print(f"Class {cls}: {count} -> augmenting {deficit} extra images to reach {majority_count}")
        augment_and_save(os.path.join(TRAIN_DIR, str(cls)), deficit)

print("\nTrain class counts (after augmentation):", count_classes(TRAIN_DIR))


Class 1: 314 -> augmenting 1220 extra images to reach 1534


100%|██████████| 1220/1220 [00:31<00:00, 38.18it/s]


Class 2: 849 -> augmenting 685 extra images to reach 1534


100%|██████████| 685/685 [00:17<00:00, 38.24it/s]


Class 3: 164 -> augmenting 1370 extra images to reach 1534


100%|██████████| 1370/1370 [00:35<00:00, 38.19it/s]


Class 4: 251 -> augmenting 1283 extra images to reach 1534


100%|██████████| 1283/1283 [00:32<00:00, 38.94it/s]


Train class counts (after augmentation): {0: 1534, 1: 1534, 2: 1534, 3: 1534, 4: 1534}


## 5. Persist to Google Drive

In [14]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/DR_data/2019"
os.makedirs(DRIVE_ROOT, exist_ok=True)

shutil.copytree(TRAIN_DIR, os.path.join(DRIVE_ROOT, "train"), dirs_exist_ok=True)
shutil.copytree(VAL_DIR, os.path.join(DRIVE_ROOT, "val"), dirs_exist_ok=True)

print("Saved to:", DRIVE_ROOT)


Mounted at /content/drive
Saved to: /content/drive/MyDrive/DR_data/2019


## Summary

- Raw images and labels pulled fresh from `train.csv` / `train_images/` -- the unlabeled `test.csv`
  is never used for anything here.
- One stratified 85/15 split with a fixed seed -- reproducible, and no image can appear in both
  train and val, which fixes the leakage risk in the old notebook.
- Preprocessed straight to 528×528, same crop + resize + unsharp mask as the 2015 notebook.
- Train split rebalanced via offline augmentation so every class matches the majority class count; val left untouched.
- Final data lives at `/content/drive/MyDrive/DR_data/2019/{train,val}`.

**Still open:** the training notebook currently points at `preprocessed_train2019/preprocessed_train`
and `preprocesses_tesstt2019` -- when we rebuild the training notebook, its paths need to be updated
to match `2019/train` and `2019/val` above.
